# Part G: Parameter Management & Model Surgery

---

## Why This Matters

You've been building and training models from scratch every session. In real projects — and in your EEND-SS system — you'll rarely do that. You'll save a model after training, load it later to continue training or run inference, freeze parts of it to train only specific layers, and load weights from pretrained models to avoid starting from zero.

These are practical engineering skills. Getting them wrong means lost training runs, corrupted checkpoints, and subtle bugs where a model silently loads the wrong weights.

---

## `.parameters()` vs `.named_parameters()`

Every `nn.Module` has two ways to iterate over its weights:

```python
model.parameters()        # iterator of tensors only
model.named_parameters()  # iterator of (name, tensor) pairs
```

`.parameters()` is what you pass to an optimizer — it just needs the tensors:

```python
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

`.named_parameters()` is what you use when you need to know *which* parameter you're looking at — for freezing specific layers, inspecting weights by name, or loading pretrained weights selectively:

```python
for name, param in model.named_parameters():
    print(name, param.shape)
```

For your `MiniEEND`, this would print something like:

```
linearprojection.weight    torch.Size([64, 80])
linearprojection.bias      torch.Size([64])
block1.attention.linear_Q.weight    torch.Size([64, 64])
block1.attention.linear_Q.bias      torch.Size([64])
...
outputlayer.weight    torch.Size([3, 64])
outputlayer.bias      torch.Size([3])
```

The names follow the attribute path you used when defining the module. `block1.attention.linear_Q.weight` means: `model.block1` → `.attention` → `.linear_Q` → `.weight`. This hierarchical naming is how you target specific layers precisely.

---

## `.state_dict()`: What It Contains

A model's `state_dict()` is an ordered dictionary mapping parameter names to their current tensor values. It contains everything needed to reproduce the model's exact state:

```python
sd = model.state_dict()
# OrderedDict([
#   ('linearprojection.weight', tensor([...])),
#   ('linearprojection.bias',   tensor([...])),
#   ('block1.attention.linear_Q.weight', tensor([...])),
#   ...
# ])
```

Two important things it contains that people miss:

**Learned parameters** — all the weights and biases you'd expect.

**Buffers** — non-trainable tensors that are part of the model state. For example, `BatchNorm` tracks running mean and variance during training. These aren't parameters (they don't get gradients) but they're part of the model's state and must be saved. `state_dict()` includes them. If you only saved parameters and not buffers, your BatchNorm layers would behave incorrectly at inference time.

---

## `state_dict()` vs `named_parameters()`

They look similar because both give you name → tensor mappings, but they serve different purposes:

```python
model.named_parameters()  # for iteration during training
model.state_dict()        # for saving/loading model state
```

**`named_parameters()`** is a live iterator over the model's trainable parameters. It gives you direct references to the actual tensors — if you modify them, you're modifying the model. It only includes parameters (things with gradients), not buffers like BatchNorm running stats. You use it when you need to inspect or manipulate weights during training — freezing, gradient checking, passing to an optimizer.

**`state_dict()`** is a complete snapshot of everything needed to reproduce the model's state — parameters AND buffers — copied into an ordered dictionary. It's for persistence: saving to disk and loading back later. The values are detached copies, not live references.

```
named_parameters()  →  live references, parameters only, used during training
state_dict()        →  detached snapshot, parameters + buffers, used for saving
```

A concrete way to feel the difference: if you do `torch.save(model.named_parameters(), ...)` you'd save an iterator object, which is useless. `torch.save(model.state_dict(), ...)` saves the actual weight values. That's why the golden rule is always `state_dict()` for saving.

---

## Saving and Loading: Two Approaches

### Approach 1: Save the full model

```python
torch.save(model, 'model.pt')
model = torch.load('model.pt')
```

Saves the entire model object including architecture. Convenient but fragile — if you rename a class or change the file structure, loading breaks because PyTorch needs the original class definition to reconstruct the object.

### Approach 2: Save only the weights (recommended)

```python
torch.save(model.state_dict(), 'weights.pt')

# To load:
model = MiniEEND(n_mels=80, d_model=64, num_speakers=3)  # recreate architecture
model.load_state_dict(torch.load('weights.pt'))
```

You recreate the architecture in code, then fill it with the saved weights. This is more robust — the weights file has no dependency on your class names or file structure. The architecture is defined in your code, which you control.

**The golden rule:** always save `state_dict()`, never the full model object.

### Checkpoint saving (what you actually want)

In real training you save more than just weights — you save everything needed to resume training:

```python
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_loss': val_loss,
}
torch.save(checkpoint, 'checkpoint.pt')

# To resume:
checkpoint = torch.load('checkpoint.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
```

The optimizer state matters because Adam tracks momentum and per-parameter learning rate history. If you only reload model weights but not optimizer state, Adam restarts from scratch and your training doesn't resume smoothly.

---

## Freezing Layers: When and Why

Every parameter has a `.requires_grad` attribute. When `True`, gradients are computed for it during backprop and the optimizer updates it. When `False`, it's frozen — the backward pass skips it entirely.

```python
param.requires_grad = False   # freeze this parameter
param.requires_grad = True    # unfreeze
```

To freeze an entire submodule:

```python
for param in model.block1.parameters():
    param.requires_grad = False
```

**When do you freeze layers?**

**During feature extraction:** You have a pretrained model that's good at extracting LMF-like representations. You want to use it as a fixed feature extractor and only train a new classifier on top. Freeze everything except the final layer.

**To stabilize training:** When fine-tuning, the pretrained layers can be fragile early in training — large gradients from your new task can destroy the pretrained representations. Freeze the early layers for the first few epochs, then unfreeze.

**To save compute:** Frozen parameters don't need gradients computed. For large pretrained models, this significantly reduces memory and compute cost.

One important detail: freezing parameters doesn't automatically affect `Dropout` or `BatchNorm` behavior. You still need `model.eval()` to disable Dropout and switch BatchNorm to inference mode. Freezing and eval mode are separate concepts that are often needed together but do different things.

---

## Transfer Learning and Fine-Tuning

These two terms are related but distinct:

**Transfer learning:** Take weights trained on task A, use them as the starting point for task B. The idea is that useful representations (like "what a voice sounds like") transfer across tasks.

**Fine-tuning:** Specifically, taking a pretrained model and continuing to train it on your new task — usually with a small learning rate so you don't destroy the pretrained knowledge.

The typical workflow:

```
Stage 1 — Feature extraction (freeze pretrained, train head only):
  - Load pretrained weights
  - Freeze all pretrained layers
  - Train only the new task-specific head
  - Run for a few epochs until the head is reasonable

Stage 2 — Fine-tuning (unfreeze some layers, train everything):
  - Unfreeze later layers of the pretrained model
  - Use a small learning rate (10-100× smaller than Stage 1)
  - Train end-to-end
  - Early layers usually stay frozen (they learn universal features)
```

Why the two stages? In Stage 1, your new head starts with random weights — it produces random gradients that would corrupt pretrained weights if they were unfrozen. You first stabilize the head, then fine-tune the whole network.

For your EEND system specifically: when you eventually build the full system, you might pretrain the transformer blocks on a large speaker recognition dataset, then fine-tune on your diarization task. The LMF extraction and initial projection layers would be frozen, the transformer blocks would be fine-tuned with a small lr, and only the diarization output head would be trained from scratch.

---

## Parameter Counting

Understanding model size matters for two reasons: knowing if your model will fit in memory, and comparing model complexity fairly between architectures.

```python
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable
```

`p.numel()` returns the total number of elements in a tensor — for a weight of shape `(64, 80)` that's `64 × 80 = 5120`.

A useful breakdown by layer:

```python
for name, param in model.named_parameters():
    print(f"{name}: {param.numel():,} params, requires_grad={param.requires_grad}")
```

Rule of thumb for memory: each float32 parameter uses 4 bytes. During training you also need gradients (another 4 bytes) and optimizer state (Adam uses 8 more bytes — momentum and variance). So each trainable parameter costs roughly **16 bytes** during training.

Your MiniEEND has roughly:
```
linearprojection:   80×64 + 64        =   5,184
block1 attention:   4 × (64×64 + 64)  =  16,640
block1 FFN:         64×256 + 256×64   =  32,768  (+biases ~768)
block2:             same as block1    =  ~50,000
outputlayer:        64×3 + 3          =     195
Total:              ~105,000 params   =  ~1.7MB in training
```

Tiny by modern standards. GPT-2 small has 117 million parameters. Your full EEND system will likely be in the 1-10M range.

---

## TODO: Parameter Management on MiniEEND

Three parts — each builds directly on the last.

**Part 1 — Inspect and count:**

- Print all named parameters of your `MiniEEND` with their shapes and `requires_grad` status
- Count total parameters, trainable parameters
- Compute the model size in KB assuming float32

**Part 2 — Save, corrupt, reload:**

- Save `MiniEEND`'s `state_dict` to disk
- Deliberately corrupt the model by reinitializing all weights: `model.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)`
- Run a forward pass and record the output (it will be different from before)
- Reload the saved `state_dict`
- Run the same forward pass again and verify the output matches the original using `torch.allclose`

This proves to you that `state_dict` fully captures model state.

**Part 3 — Freeze, train, unfreeze:**

- Take your `MiniEEND` and freeze everything except `outputlayer`
- Verify: print trainable parameter count before and after freezing
- Run one full training step with PIT loss — confirm that only `outputlayer` parameters have non-None gradients after `.backward()`
- Then unfreeze `block2` only (both attention and FFN)
- Run another training step — confirm that `block1` parameters still have None gradients but `block2` and `outputlayer` do not

**Hints:**
- To save: `torch.save(model.state_dict(), 'miniteend_weights.pt')`
- To load: `model.load_state_dict(torch.load('miniteend_weights.pt'))`
- To freeze a specific submodule: iterate `model.block1.parameters()` and set `requires_grad=False`
- After freezing, pass only trainable parameters to the optimizer: `optim.Adam(filter(lambda p: p.requires_grad, model.parameters()))`
- To check gradients per layer after backward: `model.outputlayer.weight.grad` should be non-None, `model.block1.attention.linear_Q.weight.grad` should be None
- `model.apply()` visits every submodule recursively — useful for bulk operations

**Key questions:**
- After freezing `block1` and running backward, does the frozen layer's `.grad` attribute stay `None` or does it become a zero tensor? What's the difference?
- Why do you need to pass only trainable parameters to the optimizer? What happens if you pass frozen parameters too?
- In Stage 2 fine-tuning you use a smaller learning rate than Stage 1. Why can't you just use the same learning rate?

---

Start with Part 1. Show me the full parameter table before writing any other code.

In [91]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import scipy
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as T
import math
from pathlib import Path
import numpy as np
import os
import sys
import json
import matplotlib.pyplot as plt

# Check if __file__ exists (it won't in Jupyter)
try: 
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


In [92]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        mixture_tensor = torch.from_numpy(mixture_audio)
        
        max_len = 480000 
        
        if mixture_tensor.size(0) > max_len:
            # Truncate if too long
            mixture_tensor = mixture_tensor[:max_len]
        else:
            # Pad with zeros if too short
            padding = max_len - mixture_tensor.size(0)
            mixture_tensor = torch.nn.functional.pad(mixture_tensor, (0, padding))
        
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        data = np.load(entry['label_path'], allow_pickle=True)
        audio_labels = torch.from_numpy(data)
        
        label_max_frames = 2994
        max_speakers = 3
        
        current_frames = audio_labels.shape[0]
        if current_frames < label_max_frames:
            pad_frames = label_max_frames - current_frames
            audio_labels = F.pad(audio_labels, (0, 0, 0, pad_frames))  # pad time dim
        elif current_frames > label_max_frames:
            audio_labels = audio_labels[:label_max_frames, :]
        
        current_speakers = audio_labels.shape[1]
        if current_speakers < max_speakers:
            pad_cols = max_speakers - current_speakers
            audio_labels = F.pad(audio_labels, (0, pad_cols))
        
        return mixture_tensor, label_tensor, audio_labels

In [93]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

10000
2000


In [94]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

In [95]:
def torchLMF(audio, n_fft, hop_length):
    torch_lmf = T.MelSpectrogram(
        sample_rate=16000,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=80,
        center=False
        )
    mel_spec = torch_lmf(audio)
    log = torch.log(mel_spec + 1e-8)
    return log

In [96]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.linear_Q = nn.Linear(d_model, d_model)
        self.linear_K = nn.Linear(d_model, d_model)
        self.linear_V = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.linear_Q(x)
        K = self.linear_K(x)
        V = self.linear_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)
        output = self.out_proj(context)
        return output

class TransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = SelfAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self,x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class MiniEEND(nn.Module):
    def __init__(self, n_mels, d_model, num_speakers):
        super().__init__()
        self.linearprojection = nn.Linear(n_mels, d_model)
        self.block1 = TransformerBlock(d_model=d_model)
        self.block2 = TransformerBlock(d_model=d_model)
        self.outputlayer = nn.Linear(d_model, num_speakers)
        

    def forward(self,x):
        x = x.transpose(-1,-2)
        x = self.linearprojection(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.outputlayer(x)
        x = torch.sigmoid(x)
        return x

In [97]:
def align_temporal_dimensions(features, targets):
    """
    Synchronizes the time dimension (T) between features and targets 
    by truncating to the minimum length available in both.
    """
    # Identify time dimensions based on expected shapes:
    # Features: (80, T) or (Batch, 80, T)
    # Targets: (T, Speakers) or (Batch, T, Speakers)
    T_lmf = features.shape[-1]       
    T_label = targets.shape[-2]  
    T_min = min(T_lmf, T_label)          
    
    # Handle single sample (2D features)
    if features.dim() == 2:
        features = features[..., :T_min]         
        targets = targets[:T_min, :]    
    
    # Handle batch (3D features)
    elif features.dim() == 3:
        features = features[..., :T_min]         
        targets = targets[:, :T_min, :]    
    
    return features, targets

In [98]:
n_mels = 80
d_model=64
num_speakers=3

model = MiniEEND(n_mels=n_mels, d_model=d_model, num_speakers=num_speakers)

### todo

part 1

In [99]:
for name, param in model.named_parameters():
    print(f"{name}: {param.numel():,} params, requires_grad={param.requires_grad}")

linearprojection.weight: 5,120 params, requires_grad=True
linearprojection.bias: 64 params, requires_grad=True
block1.attention.linear_Q.weight: 4,096 params, requires_grad=True
block1.attention.linear_Q.bias: 64 params, requires_grad=True
block1.attention.linear_K.weight: 4,096 params, requires_grad=True
block1.attention.linear_K.bias: 64 params, requires_grad=True
block1.attention.linear_V.weight: 4,096 params, requires_grad=True
block1.attention.linear_V.bias: 64 params, requires_grad=True
block1.attention.out_proj.weight: 4,096 params, requires_grad=True
block1.attention.out_proj.bias: 64 params, requires_grad=True
block1.norm1.weight: 64 params, requires_grad=True
block1.norm1.bias: 64 params, requires_grad=True
block1.feed_forward.0.weight: 16,384 params, requires_grad=True
block1.feed_forward.0.bias: 256 params, requires_grad=True
block1.feed_forward.2.weight: 16,384 params, requires_grad=True
block1.feed_forward.2.bias: 64 params, requires_grad=True
block1.norm2.weight: 64 para

In [100]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"total parameters: {total}")
print(f"trainable parameters: {trainable}")

total parameters: 105347
trainable parameters: 105347


In [101]:
frozen_para = total - trainable
memory = ((frozen_para*4) + (trainable*16)) / 1024

print(f"memory need to train the model in kb: {memory:.2f}")

memory need to train the model in kb: 1646.05


part 2

In [102]:
model_save_path = "models/model.pt"
torch.save(model.state_dict(), model_save_path)

In [103]:
n_fft = 400
hop_length = 160

batch, num, targets = next(iter(train_loader))

print(f"batch shape: {batch.shape}")
print(f"number of speakers shape: {num.shape}")

features = torchLMF(batch, n_fft=n_fft, hop_length=hop_length)
features, targets = align_temporal_dimensions(features, targets)

print(f"targets shape: {targets.shape}")
print(f"input shape: {features.shape}")

batch shape: torch.Size([32, 480000])
number of speakers shape: torch.Size([32])
targets shape: torch.Size([32, 2994, 3])
input shape: torch.Size([32, 80, 2994])


In [104]:
predictions_nor = model(features)

print(f"predictions shape: {predictions_nor.shape}")

predictions shape: torch.Size([32, 2994, 3])


In [105]:
model.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)

MiniEEND(
  (linearprojection): Linear(in_features=80, out_features=64, bias=True)
  (block1): TransformerBlock(
    (attention): SelfAttention(
      (linear_Q): Linear(in_features=64, out_features=64, bias=True)
      (linear_K): Linear(in_features=64, out_features=64, bias=True)
      (linear_V): Linear(in_features=64, out_features=64, bias=True)
      (out_proj): Linear(in_features=64, out_features=64, bias=True)
    )
    (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (feed_forward): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=64, bias=True)
    )
    (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (block2): TransformerBlock(
    (attention): SelfAttention(
      (linear_Q): Linear(in_features=64, out_features=64, bias=True)
      (linear_K): Linear(in_features=64, out_features=64, bias=True)
      (linear_V): Linear(in_features=64, out_features=6

In [106]:
predictions_corrupt = model(features)

print(f"predictions shape: {predictions_corrupt.shape}")

predictions shape: torch.Size([32, 2994, 3])


In [107]:
model_new = MiniEEND(n_mels=n_mels, d_model=d_model, num_speakers=num_speakers)
model_new.load_state_dict(torch.load(model_save_path))

predictions_reload = model_new(features)

print(f"predictions shape: {predictions_reload.shape}")

predictions shape: torch.Size([32, 2994, 3])


In [108]:
corrupt_compare = torch.allclose(predictions_nor, predictions_corrupt)
reload_compare = torch.allclose(predictions_nor, predictions_reload)

print(f"comparision of model and corrupt model: {corrupt_compare}")
print(f"comparision of model and reloaded model: {reload_compare}")

comparision of model and corrupt model: False
comparision of model and reloaded model: True


part 3

In [109]:
def compute_cost_matrix(predictions, targets):
    epsilon = 1e-8
    num_speakers = predictions.shape[-1]
    cost_matrix = torch.zeros(num_speakers, num_speakers)
    
    for i in range(num_speakers):
        for j in range(num_speakers):
            p = predictions[:, i]
            t = targets[:, j]
            cost_matrix[i][j] = -torch.mean(t*torch.log(p+epsilon) + (1-t)*torch.log(1-p+epsilon))
    
    return cost_matrix

def pit_loss_single(predictions, targets):
    cost_matrix = compute_cost_matrix(predictions, targets)
    row_idx, col_idx = scipy.optimize.linear_sum_assignment(cost_matrix.detach().numpy())
    loss = cost_matrix[row_idx, col_idx].mean()
    return loss

def pit_loss_batch(predictions, targets):
    if predictions.dim() == 2:
        return pit_loss_single(predictions, targets)
    
    num_batch = predictions.shape[0]
    losses = []
    
    for i in range(num_batch):
        losses.append(pit_loss_single(predictions[i], targets[i]))
    
    return torch.stack(losses).mean()

In [110]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"trainable parameters: {trainable}")

trainable parameters: 105347


In [111]:
for param in model.linearprojection.parameters():
    param.requires_grad = False

for param in model.block1.parameters():
    param.requires_grad = False

for param in model.block2.parameters():
    param.requires_grad = False

In [112]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"trainable parameters: {trainable}")

trainable parameters: 195


In [113]:
model.train()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()))

batch, num, targets = next(iter(train_loader))
features = torchLMF(batch, n_fft=n_fft, hop_length=hop_length)
features, targets = align_temporal_dimensions(features, targets)

logits = model(features)

loss = pit_loss_batch(logits, targets)
loss.backward()

print(model.block1.attention.linear_Q.weight.grad)
print(model.block2.attention.linear_Q.weight.grad)
print(model.outputlayer.weight.grad)

optimizer.step()
optimizer.zero_grad()

None
None
tensor([[-0.3488,  0.4465, -0.3407, -0.0700,  0.5269,  0.4419,  0.2029,  0.8724,
         -0.1485, -0.0209,  0.0562, -0.0923,  0.3451,  0.0241,  0.2302,  0.2095,
          0.1502, -0.0097, -0.0812,  0.0883,  0.6022,  0.5235, -0.4015,  0.0784,
         -0.1131,  0.1395,  0.2546,  0.0519,  0.0612,  0.0204,  0.3211,  0.3112,
         -0.1261,  0.3639, -0.0945,  0.2198, -0.0443,  0.5269,  0.5171,  0.2907,
         -0.5490,  0.1809, -0.4169,  0.0120,  0.0655,  0.0094, -0.1337, -0.2547,
          0.1454,  0.1529,  0.3084,  0.2855,  0.1356, -0.0983, -0.0711,  0.5929,
         -0.4042, -0.4591, -0.1172,  0.4381, -0.3450,  0.0059,  0.2622,  0.4727],
        [ 0.5678,  0.3678,  0.5137,  0.2769, -0.1267, -0.4148, -0.5332, -0.5013,
          0.0106,  0.1499, -0.0872,  0.1363, -0.4803, -0.4282, -0.4124, -0.5319,
         -0.0195,  0.3260, -0.0500,  0.3558, -0.5268, -0.5343, -0.0604,  0.2453,
         -0.3532, -0.0270, -0.3299, -0.6186, -0.1219,  0.2095,  0.0880, -0.1661,
         -0.0539,

In [114]:
for param in model.block2.parameters():
    param.requires_grad = True

In [115]:
model.train()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()))

batch, num, targets = next(iter(train_loader))
features = torchLMF(batch, n_fft=n_fft, hop_length=hop_length)
features, targets = align_temporal_dimensions(features, targets)

logits = model(features)

loss = pit_loss_batch(logits, targets)
loss.backward()

print(model.block1.attention.linear_Q.weight.grad)
print(model.block2.attention.linear_Q.weight.grad)
print(model.outputlayer.weight.grad)

optimizer.step()
optimizer.zero_grad()

None
tensor([[-1.3819e-04,  2.2063e-04, -1.7254e-04,  ..., -3.7842e-05,
          1.5874e-04,  9.0156e-05],
        [ 3.8382e-05,  9.8330e-05,  1.8075e-05,  ...,  2.6414e-05,
          5.0162e-05, -3.8913e-05],
        [ 8.1273e-05,  1.7020e-04,  5.2128e-05,  ...,  5.4391e-05,
          9.4201e-05, -7.9184e-05],
        ...,
        [-6.8885e-05,  2.1717e-04, -9.6752e-05,  ..., -1.0009e-05,
          1.4942e-04,  3.6593e-05],
        [-2.5311e-04,  2.0433e-04, -2.8385e-04,  ..., -7.6176e-05,
          1.4836e-04,  1.7923e-04],
        [ 1.3289e-04, -8.0993e-05,  1.4472e-04,  ...,  4.6757e-05,
         -6.3392e-05, -1.0112e-04]])
tensor([[-3.5391e-01,  4.8033e-01, -3.4515e-01, -7.9586e-02,  5.2801e-01,
          4.5367e-01,  2.0493e-01,  8.7436e-01, -1.5299e-01, -4.6102e-03,
          4.2931e-02, -7.9972e-02,  3.5182e-01,  1.6655e-02,  2.3329e-01,
          2.1264e-01,  1.6485e-01,  3.3095e-03, -9.6677e-02,  1.0478e-01,
          6.1603e-01,  5.1677e-01, -4.2297e-01,  9.5210e-02, -1.215